<a href="https://colab.research.google.com/github/group6aiml-hub/NIDS_AInML/blob/notebooks/notebooks/05_imbalance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Loading in CLean data
!pip install -q imbalanced-learn

from google.colab import drive
drive.mount('/content/drive')
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.metrics import f1_score, recall_score, classification_report

import time
import pandas as pd
import numpy as np

path = '/content/drive/MyDrive/Datasets/'
train = pd.read_csv(path + 'train_clean.csv')
test  = pd.read_csv(path + 'test_clean.csv')
print(train.shape, test.shape)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
(107740, 43) (82332, 43)


In [ ]:
#Skill ut X somer features og Y somer fasit

X_train = train.drop(columns=['label', 'attack_cat'])
y_train = train['attack_cat']
X_test  = test.drop(columns=['label', 'attack_cat'])
y_test  = test['attack_cat']

In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=42)
print(y_tr.value_counts())

attack_cat
Normal            41512
Exploits          15875
Fuzzers           12920
Reconnaissance     6018
Generic            3345
DoS                3045
Analysis           1275
Backdoor           1228
Shellcode           873
Worms               101
Name: count, dtype: int64


In [ ]:
#Del kolonnene i grupper

cat_cols = ['proto', 'service', 'state']
num_cols = X_tr.columns.drop(cat_cols).tolist()

skew = X_tr[num_cols].skew()
skew_cols  = skew[skew.abs() > 1].index.tolist()
other_cols = [c for c in num_cols if c not in skew_cols]

print(len(cat_cols), len(skew_cols), len(other_cols))

3 32 6


In [ ]:
# Bygg preprossesering

preprocess = ColumnTransformer([
    ('cat',  OneHotEncoder(handle_unknown='ignore'), cat_cols),
    ('skew', Pipeline([('log',   FunctionTransformer(np.log1p)),
                       ('scale', StandardScaler())]), skew_cols),
    ('num',  StandardScaler(), other_cols),
])

In [ ]:
resultater = []
sma_klasser = ['Worms', 'Shellcode', 'Backdoor', 'Analysis', 'DoS']

def kjor_eksperiment(navn, pipeline):
    start = time.time()
    pipeline.fit(X_tr, y_tr)
    tid = time.time() - start

    y_pred = pipeline.predict(X_val)
    rad = {'Eksperiment': navn,
           'Macro-F1': f1_score(y_val, y_pred, average='macro'),
           'Treningstid (s)': round(tid, 1)}
    for k, r in zip(sma_klasser, recall_score(y_val, y_pred, labels=sma_klasser, average=None)):
        rad[f'Recall {k}'] = r
    resultater.append(rad)

    print(classification_report(y_val, y_pred, digits=3, zero_division=0))
    return pipeline

def ny_rf(**kw):
    return RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, **kw)

In [ ]:
# Ingen tiltak

p1 = Pipeline([('prep', preprocess), ('clf', ny_rf())])
kjor_eksperiment('1: Ingen tiltak', p1)

                precision    recall  f1-score   support

      Analysis      0.058     0.053     0.055       319
      Backdoor      0.198     0.169     0.183       307
           DoS      0.356     0.197     0.254       761
      Exploits      0.780     0.867     0.821      3969
       Fuzzers      0.681     0.746     0.712      3230
       Generic      0.921     0.818     0.866       836
        Normal      0.940     0.921     0.930     10378
Reconnaissance      0.749     0.735     0.742      1504
     Shellcode      0.635     0.670     0.652       218
         Worms      0.625     0.192     0.294        26

      accuracy                          0.815     21548
     macro avg      0.594     0.537     0.551     21548
  weighted avg      0.810     0.815     0.811     21548



Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['proto', 'service',
                                                   'state']),
                                                 ('skew',
                                                  Pipeline(steps=[('log',
                                                                   FunctionTransformer(func=<ufunc 'log1p'>)),
                                                                  ('scale',
                                                                   StandardScaler())]),
                                                  ['dur', 'spkts', 'dpkts',
                                                   'sbytes', 'dbytes', 'rate',
                                                   'sload', 'dload', 'sloss',
                                                   'dloss', 'sinpkt', 'dinpkt',
                                                   'sjit', 'djit', 'tcprtt',
                                                   'synack', 'ackdat', 'smean',
                                                   'dmean', 'trans_depth',
                                                   'response_body_len',
                                                   'ct_srv_src', 'ct_state_ttl',
                                                   'ct_dst_ltm',
                                                   'ct_src_dport_ltm',
                                                   'ct_dst_sport_ltm',
                                                   'ct_dst_src_ltm',
                                                   'ct_ftp_cmd',
                                                   'ct_flw_http_mthd',
                                                   'ct_src_ltm', ...]),
                                                 ('num', StandardScaler(),
                                                  ['sttl', 'dttl', 'swin',
                                                   'stcpb', 'dtcpb',
                                                   'dwin'])])),
                ('clf', RandomForestClassifier(n_jobs=-1, random_state=42))])

In [ ]:
#Class weight
p2 = Pipeline([
    ('prep', preprocess),
    ('clf',  ny_rf(class_weight='balanced')),
])
kjor_eksperiment('2: class_weight', p2)

                precision    recall  f1-score   support

      Analysis      0.037     0.078     0.050       319
      Backdoor      0.085     0.195     0.119       307
           DoS      0.400     0.184     0.252       761
      Exploits      0.829     0.863     0.846      3969
       Fuzzers      0.738     0.727     0.733      3230
       Generic      0.943     0.824     0.879       836
        Normal      0.934     0.925     0.929     10378
Reconnaissance      0.877     0.734     0.799      1504
     Shellcode      0.627     0.647     0.637       218
         Worms      0.562     0.346     0.429        26

      accuracy                          0.814     21548
     macro avg      0.603     0.552     0.567     21548
  weighted avg      0.834     0.814     0.821     21548



Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['proto', 'service',
                                                   'state']),
                                                 ('skew',
                                                  Pipeline(steps=[('log',
                                                                   FunctionTransformer(func=<ufunc 'log1p'>)),
                                                                  ('scale',
                                                                   StandardScaler())]),
                                                  ['dur', 'spkts', 'dpkts',
                                                   'sbytes', 'dbytes', 'rate',
                                                   'sload', 'dload', 'sloss',
                                                   'dloss', 'sinpkt', 'dinpkt',
                                                   'sjit', 'djit', 'tcpr...
                                                   'synack', 'ackdat', 'smean',
                                                   'dmean', 'trans_depth',
                                                   'response_body_len',
                                                   'ct_srv_src', 'ct_state_ttl',
                                                   'ct_dst_ltm',
                                                   'ct_src_dport_ltm',
                                                   'ct_dst_sport_ltm',
                                                   'ct_dst_src_ltm',
                                                   'ct_ftp_cmd',
                                                   'ct_flw_http_mthd',
                                                   'ct_src_ltm', ...]),
                                                 ('num', StandardScaler(),
                                                  ['sttl', 'dttl', 'swin',
                                                   'stcpb', 'dtcpb',
                                                   'dwin'])])),
                ('clf',
                 RandomForestClassifier(class_weight='balanced', n_jobs=-1,
                                        random_state=42))])

In [ ]:
#SMOTE
smote = SMOTE(sampling_strategy={'Worms': 3000, 'Shellcode': 3000, 'Backdoor': 3000,
                                 'Analysis': 3000, 'DoS': 5000},
              random_state=42)

p3 = ImbPipeline([
    ('prep',  preprocess),
    ('smote', smote),
    ('clf',   ny_rf()),
])
kjor_eksperiment('3: SMOTE', p3)

                precision    recall  f1-score   support

      Analysis      0.085     0.103     0.093       319
      Backdoor      0.167     0.202     0.183       307
           DoS      0.354     0.230     0.279       761
      Exploits      0.796     0.855     0.824      3969
       Fuzzers      0.691     0.736     0.713      3230
       Generic      0.929     0.816     0.869       836
        Normal      0.940     0.918     0.929     10378
Reconnaissance      0.790     0.731     0.760      1504
     Shellcode      0.557     0.739     0.635       218
         Worms      0.368     0.538     0.438        26

      accuracy                          0.813     21548
     macro avg      0.568     0.587     0.572     21548
  weighted avg      0.816     0.813     0.814     21548



Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['proto', 'service',
                                                   'state']),
                                                 ('skew',
                                                  Pipeline(steps=[('log',
                                                                   FunctionTransformer(func=<ufunc 'log1p'>)),
                                                                  ('scale',
                                                                   StandardScaler())]),
                                                  ['dur', 'spkts', 'dpkts',
                                                   'sbytes', 'dbytes', 'rate',
                                                   'sload', 'dload', 'sloss',
                                                   'dloss', 'sinpkt', 'dinpkt',
                                                   'sjit', 'djit', 'tcpr...
                                                   'ct_src_dport_ltm',
                                                   'ct_dst_sport_ltm',
                                                   'ct_dst_src_ltm',
                                                   'ct_ftp_cmd',
                                                   'ct_flw_http_mthd',
                                                   'ct_src_ltm', ...]),
                                                 ('num', StandardScaler(),
                                                  ['sttl', 'dttl', 'swin',
                                                   'stcpb', 'dtcpb',
                                                   'dwin'])])),
                ('smote',
                 SMOTE(random_state=42,
                       sampling_strategy={'Analysis': 3000, 'Backdoor': 3000,
                                          'DoS': 5000, 'Shellcode': 3000,
                                          'Worms': 3000})),
                ('clf', RandomForestClassifier(n_jobs=-1, random_state=42))])

In [ ]:
#Under sampling
rus = RandomUnderSampler(sampling_strategy={'Normal': 15000}, random_state=42)

p4 = ImbPipeline([
    ('prep',  preprocess),
    ('rus',   rus),
    ('clf',   ny_rf()),
])
kjor_eksperiment('4: Undersampling', p4)

                precision    recall  f1-score   support

      Analysis      0.124     0.144     0.133       319
      Backdoor      0.192     0.163     0.176       307
           DoS      0.356     0.202     0.258       761
      Exploits      0.769     0.872     0.817      3969
       Fuzzers      0.599     0.874     0.711      3230
       Generic      0.909     0.821     0.862       836
        Normal      0.990     0.839     0.908     10378
Reconnaissance      0.742     0.731     0.737      1504
     Shellcode      0.611     0.633     0.622       218
         Worms      0.625     0.192     0.294        26

      accuracy                          0.797     21548
     macro avg      0.592     0.547     0.552     21548
  weighted avg      0.819     0.797     0.800     21548



Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['proto', 'service',
                                                   'state']),
                                                 ('skew',
                                                  Pipeline(steps=[('log',
                                                                   FunctionTransformer(func=<ufunc 'log1p'>)),
                                                                  ('scale',
                                                                   StandardScaler())]),
                                                  ['dur', 'spkts', 'dpkts',
                                                   'sbytes', 'dbytes', 'rate',
                                                   'sload', 'dload', 'sloss',
                                                   'dloss', 'sinpkt', 'dinpkt',
                                                   'sjit', 'djit', 'tcpr...
                                                   'ct_srv_src', 'ct_state_ttl',
                                                   'ct_dst_ltm',
                                                   'ct_src_dport_ltm',
                                                   'ct_dst_sport_ltm',
                                                   'ct_dst_src_ltm',
                                                   'ct_ftp_cmd',
                                                   'ct_flw_http_mthd',
                                                   'ct_src_ltm', ...]),
                                                 ('num', StandardScaler(),
                                                  ['sttl', 'dttl', 'swin',
                                                   'stcpb', 'dtcpb',
                                                   'dwin'])])),
                ('rus',
                 RandomUnderSampler(random_state=42,
                                    sampling_strategy={'Normal': 15000})),
                ('clf', RandomForestClassifier(n_jobs=-1, random_state=42))])

In [ ]:
tabell = pd.DataFrame(resultater).set_index('Eksperiment').round(3)
tabell

,Macro-F1,Treningstid (s),Recall Worms,Recall Shellcode,Recall Backdoor,Recall Analysis,Recall DoS
Eksperiment,,,,,,,
1: Ingen tiltak,0.551,39.4,0.192,0.670,0.169,0.053,0.197
2: class_weight,0.567,27.0,0.346,0.647,0.195,0.078,0.184
3: SMOTE,0.572,32.2,0.538,0.739,0.202,0.103,0.230
4: Undersampling,0.552,24.7,0.192,0.633,0.163,0.144,0.202


Tiltakene som ga høyest macro var ekspriment 3: SMOTE på 0.572, etterfult tett av ekspriment 2: Class_weight på 0.567. Etter dette kommer ekspriment 4: Undersampling en del under på 0.552, og bare da rett over eksprimentet 1 med ingen tiltak på 0.551

## Diskusjon: effekt av ubalansetiltak

Vi sammenlignet fire varianter av samme Random Forest-pipeline på valideringssettet: ingen tiltak,
`class_weight='balanced'`, SMOTE (oversampling av de fem minste klassene) og tilfeldig undersampling
av Normal til 15 000 rader.

**Totalt sett:** SMOTE ga høyest macro-F1 (0.572), fulgt av class_weight (0.567). Undersampling
(0.552) var praktisk talt lik baseline (0.551). Forbedringene er små, noe som tyder på at
klasseubalanse bare forklarer en liten del av modellens svakheter i multiklasse-oppgaven.

**Worms** fikk størst forbedring: recall økte fra 0.192 til 0.346 med class_weight og 0.538 med SMOTE.
Dette stemmer med at Worms hadde svært få treningseksempler (ca. 100). Valideringssettet inneholder
imidlertid bare 26 Worms-rader, så hvert enkelt eksempel endrer recall med ca. 0.04, og tallene er
derfor usikre.

**Analysis og Backdoor** forble svake uansett tiltak (recall maks 0.144 og 0.202). Dette støtter
hypotesen fra 04_multiclass: ca. 75 % av Analysis- og 74 % av Backdoor-radene har feature-kombinasjoner
som også forekommer med en annen fasit. Når identiske input har ulik fasit, kan ingen ubalanse-metode
gjøre klassene separerbare.

**DoS** forbedret seg lite (0.197 → 0.230). Hovedproblemet for DoS er forveksling med Exploits, ikke
mangel på eksempler, så flere DoS-rader gjør ikke klassene lettere å skille.

**Undersampling** hadde ingen effekt på de minste klassene, siden metoden bare reduserer Normal.
Den var raskest å trene, men ga ingen gevinst i macro-F1.

Økningen i recall hadde imidlertid en kostnad: precision for Worms falt fra 0.625 til 0.368 og for Shellcode fra 0.635 til 0.557, altså ga SMOTE flere falske alarmer for disse klassene. For Worms var bytteforholdet likevel positivt (F1 økte fra ca. 0.29 til 0.44), mens det for Shellcode var omtrent nøytralt.

**Begrensninger:** Resultatene er fra én kjøring med én random_state på ett valideringssett, og
forskjellene mellom tiltakene er små. Treningstidene er målt én gang i Colab og påvirkes av
oppstartskostnader, så de bør tolkes forsiktig. SMOTE interpolerer også mellom one-hot-kodede
kategoriske verdier og kan lage fysisk meningsløse rader (f.eks. `proto_tcp =

In [ ]:
tabell.to_csv(path + 'resultater_imbalance.csv')